<a href="https://colab.research.google.com/github/Emiliano23-max/Simulacion-Urbana-con-Agentes-inteligentes-usanso-el-algoritmo-A-/blob/main/simulacion_urbana_con_agentes_inteligentes__con_algoritmo_A_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install agentpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.9/53.9 kB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 800.1/800.1 kB 15.7 MB/s eta 0:00:00


In [ ]:
import agentpy as ap
import numpy as np
import math
import heapq
import random
import matplotlib.pyplot as plt
from IPython.display import HTML, display

# FUNCIONES AUXILIARES


def find_valid_position(grid, cell_type_values):
    """Busca una posición aleatoria en el grid que coincida con un tipo de celda dado."""
    rows, cols = grid.shape
    valid_coords = []
    for r in range(rows):
        for c in range(cols):
            if grid[r, c] in cell_type_values:
                valid_coords.append((r, c))
    if not valid_coords:
        raise ValueError(f"No se encontraron celdas válidas en el grid para los tipos: {cell_type_values}.")
    return random.choice(valid_coords)


# ALGORITMO A* PONDERADO BASE Y RESTRINGIDO


class WeightedAStar: #clase a* para peatones
    """A* base usado por peatones."""
    def __init__(self, grid, COST_MAP, direction_map=None):
        self.grid = grid;
        self.COST_MAP = COST_MAP;
        self.direction_map = direction_map

    def heuristic(self, a, b):
      return abs(a[0] - b[0]) + abs(a[1] - b[1])

    def get_cost(self, pos):
        x, y = pos
        if not (0 <= x < self.grid.shape[0] and 0 <= y < self.grid.shape[1]): return float('inf')
        return self.COST_MAP.get(self.grid[x, y], float('inf'))

    # get_neighbors: VECINOS ORTOGONALES ESTÁNDAR (Peatones)
    def get_neighbors(self, pos):
        x, y = pos; neighbors = []; directions = [(0, 1), (1, 0), (0, -1), (-1, 0)]
        for dx, dy in directions:
            nx, ny = x + dx, y + dy
            if (0 <= nx < self.grid.shape[0] and 0 <= ny < self.grid.shape[1]):
                if self.get_cost((nx, ny)) < float('inf'): neighbors.append((nx, ny))
        return neighbors

    def find_path(self, start, goal, weight=1.0):
        open_set = []; heapq.heappush(open_set, (0, start)); came_from = {}; g_score = {start: 0}
        f_score = {start: self.heuristic(start, goal) * weight}

        while open_set:
            current_f, current = heapq.heappop(open_set)
            if current_f > f_score.get(current, float('inf')): continue
            if current == goal:
                path = self.reconstruct_path(came_from, current)
                return path, sum(self.get_cost(pos) for pos in path[1:])

            for neighbor in self.get_neighbors(current):
                tentative_g_score = g_score.get(current, float('inf')) + self.get_cost(neighbor)

                if tentative_g_score < g_score.get(neighbor, float('inf')):
                    came_from[neighbor] = current; g_score[neighbor] = tentative_g_score
                    f_score[neighbor] = tentative_g_score + self.heuristic(neighbor, goal) * weight
                    heapq.heappush(open_set, (f_score[neighbor], neighbor))
        return None, float('inf')

    def reconstruct_path(self, came_from, current):
        path = [current];
        while current in came_from: current = came_from[current]; path.append(current)
        path.reverse(); return path

class WeightedAStarRoad(WeightedAStar):  #clase de a* para coches
    """A* para COCHES: Restringido solo a CALLE (C) y PASO ZEBRA (Z) y SENTIDO ÚNICO."""
    def __init__(self, grid, COST_MAP, allowed_values, direction_map):
        super().__init__(grid, COST_MAP, direction_map); self.allowed_values = set(allowed_values)

    def get_cost(self, pos):
        x, y = pos
        if not (0 <= x < self.grid.shape[0] and 0 <= y < self.grid.shape[1]): return float('inf')
        cell_type = self.grid[x, y]
        if cell_type not in self.allowed_values: return float('inf')
        return self.COST_MAP.get(cell_type, float('inf'))

    # get_neighbors: VECINOS RESTRINGIDOS POR SENTIDO DE CALLE (COCHES)
    def get_neighbors(self, pos):
        x, y = pos; dirs = []; direction = self.direction_map[x, y]
        cell_type = self.grid[x, y]

        if cell_type not in self.allowed_values: return []

        # logica de sentido unico
        if direction == "BOTH" or direction is None:
            candidates = [(x, y+1), (x+1, y), (x, y-1), (x-1, y)]
        elif direction == "O-E":
            candidates = [(x, y+1)] # Solo Este
        elif direction == "E-O":
            candidates = [(x, y-1)] # Solo Oeste
        elif direction == "N-S": # SOLO SUR (Abajo)
            candidates = [(x+1, y)]
        elif direction == "S-N": # SOLO NORTE (Arriba)
            candidates = [(x-1, y)]
        else:
            # Caso de respaldo
            candidates = [(x, y+1), (x, y-1), (x+1, y), (x-1, y)]

        for nx, ny in candidates:
            if 0 <= nx < self.grid.shape[0] and 0 <= ny < self.grid.shape[1]:
                if self.grid[nx, ny] in self.allowed_values and self.get_cost((nx, ny)) < float('inf'):
                    dirs.append((nx, ny))
        return dirs


# CLASE DE GENERACIÓN DEL ENTORNO


class CityGridGenerator:
    """Encapsula la lógica para crear el grid urbano con patrones y obstáculos."""

    C = 5;
    B = 1;
    Z = 2;
    E = 100

    def __init__(self, n, m, seed=None):
        self.n = n; self.m = m; self.seed = seed
        self.C = CityGridGenerator.C; self.B = CityGridGenerator.B
        self.E = CityGridGenerator.E; self.Z = CityGridGenerator.Z

        self.COST_MAP = { self.C: 5, self.B: 1, self.Z: 2, self.E: float('inf'), 50: 50, 70: 70 }
        self.BANQUETA_VALUES = [self.B, 50, 70]
        self.CALLE_VALUE = self.C
        self.CAR_VALUES = [self.C, self.Z]

        self.maze = self._generate_maze()

        self.street_direction_map = np.full((self.n, self.m), None, dtype=object)

        # LÓGICA DE SENTIDO ÚNICO CORREGIDA
        for r in range(self.n):
            for c in range(self.m):
                if self.maze[r, c] == self.C:
                    r_in_block = r % 10 # Fila dentro del bloque (10 filas)
                    c_in_block = c % 9  # Columna dentro del bloque (9 columnas)

                    # 1. Calles Horizontales (Filas 0 y 9 del bloque)
                    if r_in_block == 0 or r_in_block == 9:
                        # Alternar O-E o E-O según el índice del bloque de filas
                        if (r // 10) % 2 == 0:
                            self.street_direction_map[r, c] = "O-E" # Oeste a Este (Derecha)
                        else:
                            self.street_direction_map[r, c] = "E-O" # Este a Oeste (Izquierda)

                    # 2. Calles Verticales (Columnas 0 y 8 del bloque)
                    # Estas son las celdas C que NO son las horizontales (r_in_block 0 o 9)
                    elif c_in_block == 0 or c_in_block == 8:
                        # Alternar N-S o S-N según el índice del bloque de columnas
                        if (c // 9) % 2 == 0:
                            self.street_direction_map[r, c] = "N-S" # Norte a Sur (Abajo)
                        else:
                            self.street_direction_map[r, c] = "S-N" # Sur a Norte (Arriba)

                elif self.maze[r, c] == self.Z:
                    self.street_direction_map[r, c] = "BOTH" # Paso de cebra (Doble sentido)

    def _generate_base_block(self):
        C, B, Z, E = self.C, self.B, self.Z, self.E
        # Base block (10x9)
        base_block_data = [
            [C, Z, C, C, C, C, C, Z, C], [Z, B, B, B, B, B, B, B, Z],
            [C, B, E, E, E, E, E, B, C], [C, B, E, E, E, E, E, B, C],
            [C, B, B, B, B, B, B, B, C], [C, B, B, B, B, B, B, B, C],
            [C, B, E, E, E, E, E, B, C], [C, B, E, E, E, E, E, B, C],
            [Z, B, B, B, B, B, B, B, Z], [C, Z, C, C, C, C, C, Z, C]
        ]
        return np.array(base_block_data, dtype=int)


    def _add_random_obstacles(self, maze):
        if self.seed is not None: random.seed(self.seed)
        B = self.B; n, m = maze.shape
        valid_b_coords = []
        for r in range(n):
            for c in range(m):
                if maze[r, c] == B: valid_b_coords.append((r, c))

        # Obstáculos Costo 70
        num_obs_70 = min(10, len(valid_b_coords)); coords_70 = random.sample(valid_b_coords, num_obs_70)
        for r, c in coords_70: maze[r, c] = 70
        valid_b_coords = [coord for coord in valid_b_coords if coord not in coords_70]

        # Obstáculos Costo 50
        num_obs_50 = min(10, len(valid_b_coords)); coords_50 = random.sample(valid_b_coords, num_obs_50)
        for r, c in coords_50: maze[r, c] = 50
        return maze

    def _generate_maze(self):
        base_block = self._generate_base_block()
        block_rows, block_cols = base_block.shape
        rep_rows = math.ceil(self.n / block_rows); rep_cols = math.ceil(self.m / block_cols)
        full_maze = np.tile(base_block, (rep_rows, rep_cols))
        maze = full_maze[:self.n, :self.m]
        maze = self._add_random_obstacles(maze)
        return maze


# CLASES DE AGENTES Y GRID


class SmartAgent(ap.Agent):
    """Agente Peatón."""
    def setup(self, goal=None, astar_weight=None, **kwargs):
        self.goal = goal; self.path = []; self.current_step = 0
        self.travel_cost = 0; self.weight_used = astar_weight; self.calculated_cost = 0

    def set_path(self, path, calculated_cost, start_pos):
        self.path = path; self.calculated_cost = calculated_cost; self.current_step = 0
        self.model.env.positions[self] = start_pos

    def execute(self):
        if self.current_step < len(self.path) - 1:
            current_pos = self.get_position(); next_pos = self.path[self.current_step + 1]

            # --- LOGICA DE REPORTE DE BAJADA DE BANQUETA ---
            B_VALUES = self.model.p.BANQUETA_VALUES; C_VALUE = self.model.p.CALLE_VALUE
            current_cell_value = self.model.p.maze[current_pos]
            next_cell_value = self.model.p.maze[next_pos]



            self.model.env.positions[self] = next_pos; self.current_step += 1
            self.travel_cost += self.model.env.get_cost(next_pos); return True
        return False


    def get_position(self):
      return self.model.env.positions[self]

class SmartCarAgent(ap.Agent):
    """Agente Vehículo."""
    def setup(self, goal=None, astar_weight=None, **kwargs):
        self.goal = goal; self.path = []; self.current_step = 0
        self.travel_cost = 0; self.weight_used = astar_weight; self.calculated_cost = 0

    def set_path(self, path, calculated_cost, start_pos):
        self.path = path; self.calculated_cost = calculated_cost; self.current_step = 0
        self.model.env.positions[self] = start_pos

    def execute(self):
        if self.current_step < len(self.path) - 1:
            next_pos = self.path[self.current_step + 1]
            self.model.env.positions[self] = next_pos; self.current_step += 1
            self.travel_cost += self.model.env.get_cost(next_pos); return True
        return False

    def get_position(self):
      return self.model.env.positions[self]

class SmartGrid(ap.Grid):
    def setup(self):
      self.environment = np.copy(self.p.maze)

    def get_cost(self, pos):
      return self.p.COST_MAP.get(self.environment[pos], float('inf'))


# MODELO MULTIMODAL CON FLUJO DE PEATONES Y COCHES


class MultiModalModel(ap.Model):
    """Modelo central con flujo dinámico de peatones y coches."""
    def setup(self):
        # 1) Inicialización del entorno


        self.env = SmartGrid(self, shape=self.p.maze.shape)
        allowed_car_tiles = self.p.CAR_VALUES
        direction_map = self.p.street_direction_map

        self.astar_peds = WeightedAStar(self.p.maze, self.p.COST_MAP, direction_map)
        self.astar_cars = WeightedAStarRoad(self.p.maze, self.p.COST_MAP, allowed_values=allowed_car_tiles, direction_map=direction_map)

        # 2) Peatones Iniciales
        self.ped_agents = ap.AgentList(self, self.p.num_agents, SmartAgent)
        if self.p.num_agents > 0:
             self.env.add_agents(self.ped_agents, positions=self.p.init_positions)

        for agent, goal, weight, start_pos in zip(self.ped_agents, self.p.goals, self.p.weights_ped, self.p.init_positions):
            agent.goal = goal; agent.weight_used = weight
            path, calculated_cost = self.astar_peds.find_path(start_pos, goal, weight)

            if path: agent.set_path(path, calculated_cost, start_pos)
            else: agent.path = []

        self.active_peds = ap.AgentList(self, [a for a in self.ped_agents if a.path])

        # 3) Coches Iniciales
        self.car_agents = ap.AgentList(self, self.p.num_cars, SmartCarAgent)
        self.env.add_agents(self.car_agents, positions=self.p.car_init_positions)

        for car, goal, weight, start_pos in zip(self.car_agents, self.p.car_goals, self.p.weights_car, self.p.car_init_positions):
            car.goal = goal; car.weight_used = weight
            path, calculated_cost = self.astar_cars.find_path(start_pos, goal, weight)

            if path: car.set_path(path, calculated_cost, start_pos)
            else:
                print(f" ADVERTENCIA: Coche {car.id} NO PUDO ENCONTRAR RUTA de {start_pos} a {goal}.")
                car.path = [] # Asegura que el coche no se mueva.


        self.active_cars = ap.AgentList(self, [c for c in self.car_agents if c.path])


    def generate_new_pedestrian(self):
        """Genera y añade un nuevo Peatón con una RUTA ÚNICA."""

        weight = random.choice(self.p.weights_ped)
        used_positions = set(self.env.positions.values()); BANQUETA_VALUES = self.p.BANQUETA_VALUES
        maze = self.p.maze

        while True:
            init_pos = find_valid_position(maze, BANQUETA_VALUES)
            if init_pos not in used_positions: break

        while True:
            goal_pos = find_valid_position(maze, BANQUETA_VALUES)
            if goal_pos != init_pos and goal_pos not in used_positions: break

        new_agent = ap.AgentList(self, 1, SmartAgent); agent = new_agent[0]
        self.env.add_agents(new_agent, positions=[init_pos])

        agent.goal = goal_pos; agent.weight_used = weight
        path, calculated_cost = self.astar_peds.find_path(init_pos, goal_pos, weight)

        if path:
            agent.set_path(path, calculated_cost, init_pos)
            self.ped_agents.append(agent); self.active_peds.append(agent)
            return True
        else:
            self.env.remove_agents(agent); return False


    def generate_new_car(self):
        """Genera y añade un nuevo Coche con una RUTA ÚNICA."""

        weight = random.choice(self.p.weights_car)
        used_positions = set(self.env.positions.values()); CAR_VALUES = self.p.CAR_VALUES
        maze = self.p.maze

        while True:
            init_pos = find_valid_position(maze, CAR_VALUES)
            if init_pos not in used_positions: break

        while True:
            goal_pos = find_valid_position(maze, CAR_VALUES)
            if goal_pos != init_pos and goal_pos not in used_positions: break

        new_car = ap.AgentList(self, 1, SmartCarAgent); car = new_car[0]
        self.env.add_agents(new_car, positions=[init_pos])

        car.goal = goal_pos; car.weight_used = weight
        path, calculated_cost = self.astar_cars.find_path(init_pos, goal_pos, weight)

        if path:
            car.set_path(path, calculated_cost, init_pos)
            self.car_agents.append(car); self.active_cars.append(car)
            return True
        else:
            self.env.remove_agents(car); return False


    def step(self):
        # GENERACION DE AGENTES
        if random.random() < self.p.ped_generation_prob: self.generate_new_pedestrian()
        if random.random() < self.p.car_generation_prob: self.generate_new_car()

        # Ejecutar movimientos
        if self.active_peds: self.active_peds.execute()
        if self.active_cars: self.active_cars.execute()


    def update(self):
        # Lógica de llegadas y remoción
        def process_arrivals(active_list, is_pedestrian=True):
            newly_inactive = []
            for agent in active_list:
                if agent.get_position() == agent.goal:
                    newly_inactive.append(agent)

            if newly_inactive:
                new_active_list = ap.AgentList(self, [a for a in active_list if a not in newly_inactive])

                if new_active_list or self.active_cars and not is_pedestrian or self.active_peds and is_pedestrian:
                    try: self.env.remove_agents(newly_inactive)
                    except KeyError: pass

                return new_active_list
            return active_list

        self.active_peds = process_arrivals(self.active_peds, is_pedestrian=True)
        self.active_cars = process_arrivals(self.active_cars, is_pedestrian=False)

        if not self.active_peds and not self.active_cars:
            self.stop()


# FUNCIÓN DE ANIMACIÓN MULTIMODAL

def create_animation_plot_multimodal():
    """Dibuja peatones (rojo) y coches (azul) simultáneamente sin pintar metas."""
    def animation_plot(model, ax):
        grid = np.copy(model.p.maze)

        # Constantes de dibujo
        C, B, Z, E = model.p.C, model.p.B, model.p.Z, model.p.E
        ped_val, car_val = -1, -3  # Marcadores de agentes

        total_cost = 0
        total_peds = len(getattr(model, 'active_peds', []))
        total_cars = len(getattr(model, 'active_cars', []))

        # Peatones y Coches: Dibujo de Agentes
        #for agent in model.ped_agents:
        #    if agent in model.active_peds: grid[agent.get_position()] = ped_val
        #    total_cost += agent.travel_cost

        #for car in model.car_agents:
        #    if car in model.active_cars: grid[car.get_position()] = car_val
        #    total_cost += car.travel_cost
        #print(grid)
        # Colores
        color_dict = {
            C: '#2F2F2F',
            B: '#E8E8E8',
            E: '#8B4513',
            Z: '#F5F227',
            50: '#FFA500',
            70: '#FF4500',
            ped_val: '#FF0000',
            car_val: '#0000FF',
            # Dummy colors for safety
            -2: '#00FF00',
            -4: '#00FFFF'
        }

        ax.clear()
        img = np.zeros((grid.shape[0], grid.shape[1], 3))
        for i in range(grid.shape[0]):
            for j in range(grid.shape[1]):
                cell_val = grid[i, j]
                color_name = color_dict.get(cell_val, '#000000')
                if color_name == '#2F2F2F': color = [0.18, 0.18, 0.18]
                elif color_name == '#E8E8E8': color = [0.91, 0.91, 0.91]
                elif color_name == '#8B4513': color = [0.55, 0.27, 0.07]
                elif color_name == '#F5F227': color = [0.96, 0.95, 0.15]
                elif color_name == '#FFA500': color = [1, 0.65, 0]
                elif color_name == '#FF4500': color = [1, 0.27, 0]
                elif color_name == '#FF0000': color = [1, 0, 0]
                elif color_name == '#0000FF': color = [0, 0, 1]
                else: color = [0, 0, 0]
                img[i, j] = color

        # Peatones y Coches: Dibujo de Agentes
        for agent in model.ped_agents:
            if agent in model.active_peds:
                total_cost += agent.travel_cost
                pos = model.env.positions[agent]
                ax.plot(pos[1], pos[0], marker='o', markersize = 10, color='#FF0000')     # Agent plot

        for agent in model.car_agents:
            if agent in model.active_cars:
                total_cost += agent.travel_cost
                pos = model.env.positions[agent]
                ax.plot(pos[1], pos[0], marker='o', markersize = 10, color='#0000FF')     # Agent plot

        ax.imshow(img)
        ax.set_title(
            f"Simulación Multimodal con Flujo Continuo\n"
            f"Tiempo: {model.t} | Peatones activos: {total_peds} | Coches activos: {total_cars}\n"
            f"Costo acumulado: {total_cost:.1f}"
        )
        ax.set_xticks([]); ax.set_yticks([])

    return animation_plot


# EJECUCIÓN PRINCIPAL MULTIMODAL

# Constantes de Visualización
explorer = -1
goal_val = -2

# 1. Definir parámetros de simulación
n = 30
m = 27
SEED = 0

# --- PEATONES ---
num_agents = 15
weights_to_ped = [1.0] * num_agents
PED_GEN_PROB = 10
# --- COCHES ---
num_cars = 5
weights_to_car = [1.0] * num_cars
CAR_GEN_PROB = 1000

# Generar Entorno
random.seed(SEED)
generator = CityGridGenerator(n=n, m=m, seed=SEED)
maze = generator.maze
BANQUETA_VALUES = generator.BANQUETA_VALUES
CAR_VALUES = generator.CAR_VALUES

# Generar posiciones INICIALES (Solo si num_agents > 0)
agent_goals = []; agent_inits = []
if num_agents > 0:
    agent_inits = [find_valid_position(maze, BANQUETA_VALUES) for _ in range(num_agents)]
    agent_goals = [find_valid_position(maze, BANQUETA_VALUES) for _ in range(num_agents)]

used_positions = set()
car_init_positions = []; car_goals = []

while len(car_init_positions) < num_cars:
    pos = find_valid_position(maze, CAR_VALUES)
    if pos not in used_positions: car_init_positions.append(pos); used_positions.add(pos)

while len(car_goals) < num_cars:
    pos = find_valid_position(maze, CAR_VALUES)
    if (pos not in used_positions): car_goals.append(pos); used_positions.add(pos)

print(f"DEBUG: Coches iniciales generados: {len(car_init_positions)}")
print(f"DEBUG: Metas de coches generadas: {len(car_goals)}")

# 2. Parámetros del modelo multimodal
multi_params = {
    'seed': SEED, 'maze': maze, 'steps': 30,
    'ped_generation_prob': PED_GEN_PROB,
    'car_generation_prob': CAR_GEN_PROB,
    'num_agents': num_agents, 'init_positions': agent_inits, 'goals': agent_goals,
    'weights_ped': weights_to_ped,
    'num_cars': num_cars, 'car_init_positions': car_init_positions, 'car_goals': car_goals,
    'weights_car': weights_to_car,
    'COST_MAP': generator.COST_MAP, 'BANQUETA_VALUES': generator.BANQUETA_VALUES,
    'CALLE_VALUE': generator.CALLE_VALUE, 'CAR_VALUES': CAR_VALUES,
    'C': generator.C, 'B': generator.B, 'Z': generator.Z, 'E': generator.E,
    'explorer': explorer, 'goal_val': goal_val,
    'street_direction_map': generator.street_direction_map,
}

print("\n=== SIMULACIÓN MULTIMODAL CON FLUJO CONTINUO (Peatones y Coches) ===")
multi_model = MultiModalModel(multi_params)

fig2, ax2 = plt.subplots(figsize=(10, 10))
anim2 = ap.animate(multi_model, fig2, ax2, create_animation_plot_multimodal())
plt.close(fig2)

try:
    html2 = anim2.to_jshtml()
    display(HTML(html2))
except Exception as e:
    print(f"Error en animación multimodal: {e}")
    multi_model.run()

print("Simulación multimodal finalizada.")

DEBUG: Coches iniciales generados: 5
DEBUG: Metas de coches generadas: 5

=== SIMULACIÓN MULTIMODAL CON FLUJO CONTINUO (Peatones y Coches) ===
 ADVERTENCIA: Coche 17 NO PUDO ENCONTRAR RUTA de (9, 22) a (28, 26).
 ADVERTENCIA: Coche 18 NO PUDO ENCONTRAR RUTA de (29, 11) a (20, 25).
 ADVERTENCIA: Coche 19 NO PUDO ENCONTRAR RUTA de (10, 11) a (4, 8).
 ADVERTENCIA: Coche 20 NO PUDO ENCONTRAR RUTA de (10, 20) a (3, 9).
 ADVERTENCIA: Coche 21 NO PUDO ENCONTRAR RUTA de (8, 17) a (16, 18).


Simulación multimodal finalizada.
